In [ ]:
# -*- coding: utf-8 -*-
"""
Fake News Detection Project - Optimized Version
"""

# ======================
# 1. INSTALL REQUIREMENTS
# ======================
# Run these commands first:
# pip install numpy pandas matplotlib seaborn scikit-learn nltk wordcloud joblib tqdm

# ======================
# 2. IMPORT LIBRARIES
# ======================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import joblib
import os
from wordcloud import WordCloud
from tqdm import tqdm
tqdm.pandas()

# NLP Libraries
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# ML Libraries
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ======================
# 3. CONFIGURATION
# ======================
# Set these flags as needed
USE_SAMPLE_DATA = True      # Set to False for full dataset processing
SAMPLE_SIZE = 100         # Number of samples to use from each dataset
DISABLE_PROGRESS_BAR = True # Set to False to see progress bars

# ======================
# 4. NLTK DATA SETUP
# ======================
def setup_nltk():
    try:
        nltk.data.find('tokenizers/punkt')
        nltk.data.find('corpora/stopwords')
        nltk.data.find('corpora/wordnet')
    except LookupError:
        print("Downloading NLTK data...")
        nltk.download('punkt')
        nltk.download('stopwords')
        nltk.download('wordnet')
    
    # Set custom NLTK data path if needed
    nltk_data_path = os.path.join(os.path.expanduser("~"), "nltk_data")
    if not os.path.exists(nltk_data_path):
        os.makedirs(nltk_data_path)
    nltk.data.path.append(nltk_data_path)

setup_nltk()

# ======================
# 5. LOAD AND PREPARE DATA
# ======================
print("Loading datasets...")
try:
    true_news = pd.read_csv('True.csv')
    fake_news = pd.read_csv('Fake.csv')
    
    if USE_SAMPLE_DATA:
        print(f"Using sample data ({SAMPLE_SIZE} from each dataset)")
        true_news = pd.concat([true_news.head(SAMPLE_SIZE), true_news.tail(SAMPLE_SIZE)])
        fake_news = pd.concat([fake_news.head(SAMPLE_SIZE), fake_news.tail(SAMPLE_SIZE)])
    
except FileNotFoundError as e:
    raise FileNotFoundError(
        "Dataset files not found. Please download from Kaggle and place "
        "'True.csv' and 'Fake.csv' in your working directory"
    ) from e

# Add labels and combine
true_news['label'] = 0
fake_news['label'] = 1
news = pd.concat([true_news, fake_news], axis=0)

# Shuffle and combine text
news = news.sample(frac=1, random_state=42).reset_index(drop=True)
news['content'] = news['title'] + ' ' + news['text']

# ======================
# 6. OPTIMIZED TEXT PREPROCESSING
# ======================
def preprocess_text(text):
    """Enhanced text cleaning with multiple fallback options"""
    if not isinstance(text, str):
        return ""
    
    try:
        text = text.lower()
        text = re.sub(r'http\S+|www\S+|https\S+', '', text)
        text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
        text = re.sub(r'\d+', '', text)  # Remove numbers
        
        # Tokenize with fallback to simple split
        try:
            words = word_tokenize(text)
        except:
            words = text.split()
        
        # Stopwords removal with multiple fallbacks
        stop_words = None
        try:
            stop_words = set(stopwords.words('english'))
        except:
            try:
                stop_words = ENGLISH_STOP_WORDS
            except:
                stop_words = set()  # Empty set if all fail
        
        words = [w for w in words if w not in stop_words]
        
        # Lemmatization with fallback
        try:
            lemmatizer = WordNetLemmatizer()
            words = [lemmatizer.lemmatize(w) for w in words]
        except:
            pass  # Skip lemmatization if fails
        
        return ' '.join(words)
    except Exception as e:
        print(f"Error processing text: {str(e)[:100]}...")
        return ""

print("Preprocessing text...")
if DISABLE_PROGRESS_BAR:
    news['cleaned_content'] = news['content'].apply(preprocess_text)
else:
    news['cleaned_content'] = news['content'].progress_apply(preprocess_text)

# ======================
# 7. EXPLORATORY ANALYSIS
# ======================
print("\n=== Dataset Overview ===")
print(f"Total samples: {len(news)}")
print(f"Real news: {len(news[news['label']==0])}")
print(f"Fake news: {len(news[news['label']==1])}")

plt.figure(figsize=(10, 5))
sns.countplot(x='label', data=news)
plt.title('News Distribution (0=Real, 1=Fake)')
plt.show()

def plot_wordcloud(text, title):
    wc = WordCloud(width=800, height=400, background_color='white').generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wc)
    plt.axis('off')
    plt.title(title)
    plt.show()

print("\nGenerating word clouds...")
plot_wordcloud(' '.join(news[news['label']==0]['cleaned_content']), "Real News Word Cloud")
plot_wordcloud(' '.join(news[news['label']==1]['cleaned_content']), "Fake News Word Cloud")

# ======================
# 8. MODEL TRAINING
# ======================
print("\n=== Model Training ===")
X_train, X_test, y_train, y_test = train_test_split(
    news['cleaned_content'],
    news['label'],
    test_size=0.2,
    random_state=42
)

print("Vectorizing text...")
tfidf = TfidfVectorizer(max_df=0.7, stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training classifier...")
model = PassiveAggressiveClassifier(max_iter=50, random_state=42)
model.fit(X_train_tfidf, y_train)

# ======================
# 9. MODEL EVALUATION
# ======================
print("\n=== Model Evaluation ===")
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2%}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Real', 'Fake']))

plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, y_pred), 
            annot=True, fmt='d', 
            xticklabels=['Real', 'Fake'],
            yticklabels=['Real', 'Fake'])
plt.title('Confusion Matrix')
plt.show()

# ======================
# 10. SAVE MODEL
# ======================
print("\nSaving model...")
joblib.dump(model, 'fake_news_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

# ======================
# 11. PREDICTION FUNCTION
# ======================
def predict_news(text):
    """Predict whether news is fake or real"""
    try:
        model = joblib.load('fake_news_model.pkl')
        tfidf = joblib.load('tfidf_vectorizer.pkl')
        
        cleaned = preprocess_text(text)
        if not cleaned.strip():
            return {"error": "Text preprocessing failed"}
            
        vec = tfidf.transform([cleaned])
        pred = model.predict(vec)[0]
        
        # Get decision function scores as proxy for confidence
        if hasattr(model, 'decision_function'):
            scores = model.decision_function(vec)
            confidence = 1/(1+np.exp(-scores[0]))  # Sigmoid to get pseudo-probability
            fake_prob = confidence if pred == 1 else 1-confidence
            real_prob = 1-confidence if pred == 1 else confidence
        else:
            # Fallback for classifiers without decision_function
            confidence = 0.8  # Default confidence
            fake_prob = 0.8 if pred == 1 else 0.2
            real_prob = 0.2 if pred == 1 else 0.8
        
        return {
            'prediction': 'Fake' if pred == 1 else 'Real',
            'confidence': f"{max(fake_prob, real_prob):.1%}",
            'fake_prob': f"{fake_prob:.1%}",
            'real_prob': f"{real_prob:.1%}"
        }
    except Exception as e:
        return {"error": str(e)}

# Test prediction
test_samples = [
    "Scientists confirm climate change is accelerating with new data showing record temperatures",
    "Breaking: Alien spacecraft spotted over White House according to Pentagon sources",
    "New study finds drinking coffee may extend lifespan by 10 years"
]

print("\nTesting prediction function...")
for sample in test_samples:
    print(f"\nSample: {sample}")
    print("Result:", predict_news(sample))

print("\n=== Process Completed Successfully ===")

In [ ]:
# ======================
# TESTING SECTION
# ======================
test_cases = [
    # Clearly Fake News
    ("BREAKING: Celebrities injecting alien DNA for immortality", "Fake"),
    ("5G towers causing COVID-19, scientists admit", "Fake"),
    
    # Clearly Real News
    ("Fed raises interest rates by 0.25 percentage points", "Real"), 
    ("NASA announces new Mars rover mission", "Real"),
    
    # Borderline Cases
    ("Stock market may crash next week, analysts warn", None),
    ("Popular celebrity couple divorces after short marriage", None),
    
    # Short Headlines
    ("Free money giveaway!", "Fake"),
    ("Parliament passes new bill", "Real")
]

def test_model():
    print("\n=== Testing Model ===")
    print(f"{'Text':<50} | {'Prediction':<10} | {'Confidence':<10} | {'Expected':<10}")
    print("-"*85)
    
    for text, expected in test_cases:
        result = predict_news(text)
        
        if "error" in result:
            print(f"{text[:45]+'...':<50} | {'ERROR':<10} | {'N/A':<10} | {str(expected):<10}")
        else:
            print(f"{text[:45]+'...':<50} | {result['prediction']:<10} | {result['confidence']:<10} | {str(expected):<10}")

# Run tests
test_model()